# ResNet-50 — skin disease classification

Same workflow as `cnn_skin_disease_custom.ipynb`, but the classifier is **torchvision `resnet50`** pretrained on **ImageNet**, with the final **fully connected layer** replaced for **`NUM_CLASSES`** disease labels.

- Load images from `DatasetSkinDisease` **`train/`** / **`test/`** folders.
- **Label** = parent folder name (snake_case), **`class_id`** for PyTorch.

Run with the working directory set to **`DeepLearningProject`**.


In [ ]:
from __future__ import annotations

import re
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".JPG", ".JPEG", ".png", ".PNG"}


def folder_to_label(folder_name: str) -> str:
    s = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", folder_name)
    return s.replace("__", "_").lower()


def find_dataset_root(start: Path | None = None) -> Path:
    start = start or Path.cwd()
    candidates = [
        start / "DatasetSkinDisease" / "SkinDisease" / "SkinDisease",
        start / "DatasetSkinDisease",
        start,
    ]
    for c in candidates:
        if (c / "train").is_dir() and (c / "test").is_dir():
            return c.resolve()
    raise FileNotFoundError(
        "Could not find train/ and test/. Set DATASET_ROOT manually."
    )


DATASET_ROOT = find_dataset_root()
TRAIN_DIR = DATASET_ROOT / "train"
TEST_DIR = DATASET_ROOT / "test"
print("DATASET_ROOT:", DATASET_ROOT)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    print(
        f"GPU: {torch.cuda.get_device_name(0)} | CUDA {torch.version.cuda} | {torch.__version__}"
    )
else:
    print(f"CPU | {torch.__version__}")


In [ ]:
def collect_split(split_dir: Path, split_name: str) -> pd.DataFrame:
    rows: list[dict] = []
    if not split_dir.is_dir():
        raise FileNotFoundError(split_dir)
    for class_dir in sorted(p for p in split_dir.iterdir() if p.is_dir()):
        folder_name = class_dir.name
        label = folder_to_label(folder_name)
        for f in class_dir.iterdir():
            if f.is_file() and f.suffix in IMAGE_SUFFIXES:
                rows.append(
                    {
                        "path": f.resolve(),
                        "path_str": str(f.resolve()),
                        "split": split_name,
                        "folder_name": folder_name,
                        "label": label,
                    }
                )
    return pd.DataFrame(rows)


train_df = collect_split(TRAIN_DIR, "train")
test_df = collect_split(TEST_DIR, "test")
manifest = pd.concat([train_df, test_df], ignore_index=True)

print("Train:", len(train_df), "| Test:", len(test_df), "| Total:", len(manifest))


In [ ]:
class_names = sorted(train_df["label"].unique())
label_to_id = {lab: i for i, lab in enumerate(class_names)}

train_df = train_df.copy()
test_df = test_df.copy()
train_df["class_id"] = train_df["label"].map(label_to_id)
test_df["class_id"] = test_df["label"].map(label_to_id)

missing_test = test_df["class_id"].isna().sum()
if missing_test:
    raise ValueError(
        f"{missing_test} test labels missing from train; check folder names."
    )

train_df["class_id"] = train_df["class_id"].astype("int64")
test_df["class_id"] = test_df["class_id"].astype("int64")
manifest = pd.concat([train_df, test_df], ignore_index=True)

NUM_CLASSES = len(class_names)
print("NUM_CLASSES:", NUM_CLASSES)


In [ ]:
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

PIN_MEMORY = device.type == "cuda"


class SkinManifestDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform):
        self.paths = df["path"].tolist()
        self.targets = df["class_id"].astype(int).tolist()
        self.transform = transform

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int):
        from PIL import Image

        img = Image.open(self.paths[idx]).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        return img, self.targets[idx]


IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
IMG_SIZE = 224

train_tfms = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(
            brightness=0.15, contrast=0.15, saturation=0.1, hue=0.05
        ),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]
)
eval_tfms = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]
)

BATCH_SIZE = 32
NUM_WORKERS = 0

train_loader = DataLoader(
    SkinManifestDataset(train_df, train_tfms),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)
test_loader = DataLoader(
    SkinManifestDataset(test_df, eval_tfms),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

print(
    "train batches:", len(train_loader), "| test batches:", len(test_loader)
)


## ResNet-50 training

We load **`resnet50`** with **ImageNet** weights (`ResNet50_Weights.IMAGENET1K_V2`), replace **`model.fc`** with a linear layer mapping **2048** features to **`NUM_CLASSES`**, then train with the same loss, optimizer, scheduler, mixed precision, best checkpoint, and early stopping as the custom CNN notebook.


In [ ]:
from copy import deepcopy

from tqdm.auto import tqdm
from torchvision.models import resnet50, ResNet50_Weights

# --- hyperparameters (tune these) ---
EPOCHS = 10
LR = 3e-4
WEIGHT_DECAY = 0.05
EARLY_STOP_PATIENCE = 5

torch.manual_seed(42)
if device.type == "cuda":
    torch.cuda.manual_seed_all(42)

model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2, min_lr=1e-6
)

use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)


def train_one_epoch() -> float:
    model.train()
    total_loss = 0.0
    n = 0
    for images, labels in tqdm(train_loader, desc="train", leave=False):
        images = images.to(device, non_blocking=PIN_MEMORY)
        labels = labels.to(device, non_blocking=PIN_MEMORY)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(
            device_type=device.type, dtype=torch.float16, enabled=use_amp
        ):
            logits = model(images)
            loss = criterion(logits, labels)
        if use_amp:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * labels.size(0)
        n += labels.size(0)
    return total_loss / max(n, 1)


@torch.no_grad()
def evaluate() -> tuple[float, float]:
    model.eval()
    correct = 0
    total = 0
    run_loss = 0.0
    for images, labels in tqdm(test_loader, desc="test", leave=False):
        images = images.to(device, non_blocking=PIN_MEMORY)
        labels = labels.to(device, non_blocking=PIN_MEMORY)
        with torch.autocast(
            device_type=device.type, dtype=torch.float16, enabled=use_amp
        ):
            logits = model(images)
            loss = criterion(logits, labels)
        run_loss += loss.item() * labels.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == labels).sum().item()
        total += labels.size(0)
    acc = correct / max(total, 1)
    avg_loss = run_loss / max(total, 1)
    return avg_loss, acc


best_te_loss = float("inf")
best_state = deepcopy(model.state_dict())
best_epoch = 0
epochs_no_improve = 0

for epoch in range(1, EPOCHS + 1):
    tr_loss = train_one_epoch()
    te_loss, te_acc = evaluate()
    scheduler.step(te_loss)
    lr_now = optimizer.param_groups[0]["lr"]
    print(
        f"epoch {epoch}/{EPOCHS}  lr={lr_now:.2e}  train_loss={tr_loss:.4f}  "
        f"test_loss={te_loss:.4f}  test_acc={te_acc:.4f}"
    )
    if te_loss < best_te_loss - 1e-5:
        best_te_loss = te_loss
        best_state = deepcopy(model.state_dict())
        best_epoch = epoch
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
    if epochs_no_improve >= EARLY_STOP_PATIENCE:
        print(
            f"Early stop at epoch {epoch} (best epoch {best_epoch}, test_loss={best_te_loss:.4f})"
        )
        break

model.load_state_dict(best_state)
print(f"Using best weights from epoch {best_epoch} (test_loss={best_te_loss:.4f})")


## Confusion matrix (test set)

The next cell runs the trained **ResNet-50** `model` on `test_loader` and plots the confusion matrix (rows = true class, columns = predicted class).


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix


@torch.no_grad()
def collect_test_predictions():
    model.eval()
    y_true: list[int] = []
    y_pred: list[int] = []
    for images, labels in tqdm(test_loader, desc="predict", leave=False):
        images = images.to(device, non_blocking=PIN_MEMORY)
        labels = labels.to(device, non_blocking=PIN_MEMORY)
        with torch.autocast(
            device_type=device.type, dtype=torch.float16, enabled=use_amp
        ):
            logits = model(images)
        pred = logits.argmax(dim=1)
        y_true.extend(labels.cpu().numpy().tolist())
        y_pred.extend(pred.cpu().numpy().tolist())
    return np.asarray(y_true), np.asarray(y_pred)


y_true, y_pred = collect_test_predictions()
cm = confusion_matrix(
    y_true, y_pred, labels=np.arange(NUM_CLASSES, dtype=int)
)

fig, ax = plt.subplots(figsize=(16, 14))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm, display_labels=class_names
)
disp.plot(ax=ax, cmap="Blues", colorbar=True, xticks_rotation=90, values_format="d")
ax.set_title("ResNet-50 — confusion matrix (test set; true = row, pred = column)")
plt.tight_layout()
plt.show()
